In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_experimental.agents import create_pandas_dataframe_agent
import pandas as pd

def create_sample_data(path: str):
    """Creates a sample sales dataset for demo."""
    import random
    from datetime import date, timedelta

    random.seed(42)
    rows = []
    products = ["Laptop", "Phone", "Tablet", "Monitor", "Keyboard"]
    regions = ["North", "South", "East", "West"]
    start = date(2024, 1, 1)

    for i in range(200):
        d = start + timedelta(days=random.randint(0, 364))
        rows.append({
            "date": d.isoformat(),
            "product": random.choice(products),
            "region": random.choice(regions),
            "quantity": random.randint(1, 20),
            "unit_price": round(random.uniform(50, 2000), 2),
            "revenue": 0,
        })

    df = pd.DataFrame(rows)
    df["revenue"] = df["quantity"] * df["unit_price"]
    df.to_csv(path, index=False)
    return df

/var/folders/r2/ww1nfn0x20j21k3nmn17w4900000gn/T/ipykernel_48323/1966044361.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents import create_pandas_dataframe_agent


In [3]:
df = create_sample_data("/Users/linqibin/Documents/coding/practise_agents/apps/backend/documents/csv/sample_data.csv")

In [12]:
import os

model = os.getenv('MODEL_NAME', '')

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=model, temperature=0.3, extra_body={'enabel_thinking': False})

agent = create_pandas_dataframe_agent(llm=llm, df=df, verbose=True, allow_dangerous_code=True, agent_type="openai-tools")

In [20]:
# for chunk in agent.stream(input='"每个月的销售额趋势是怎样的？"'):
#     print(chunk)

agent.invoke(input='每个月的销售额趋势是怎样的？')



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df.groupby('month')['revenue'].sum().sort_index()"}`


month
2024-01    188879.24
2024-02    212894.75
2024-03    142107.91
2024-04    128899.83
2024-05    277043.09
2024-06    133285.90
2024-07    172698.66
2024-08    199413.54
2024-09    191502.93
2024-10    199078.42
2024-11    150621.15
2024-12    223394.64
Freq: M, Name: revenue, dtype: float64根据数据，2024年每月的销售额（revenue）趋势如下：

| 月份 | 销售额 |
|------|--------|
| 2024-01 | 188,879.24 |
| 2024-02 | 212,894.75 |
| 2024-03 | 142,107.91 |
| 2024-04 | 128,899.83 |
| 2024-05 | 277,043.09 |
| 2024-06 | 133,285.90 |
| 2024-07 | 172,698.66 |
| 2024-08 | 199,413.54 |
| 2024-09 | 191,502.93 |
| 2024-10 | 199,078.42 |
| 2024-11 | 150,621.15 |
| 2024-12 | 223,394.64 |

**趋势分析：**

1. **年初表现强劲**：1月和2月销售额较高，2月达到212,894.75。
2. **春季下滑**：3-4月销售额明显下降，4月跌至全年最低点（128,899.83）。
3. **5月峰值**：5月销售额大幅攀升至全年最高（277,043.09）。
4. **年中波动**：6月再次大幅下降，随后7-10月保持在17万-20万之间的稳定水平。
5. **年末回升**：1

{'input': '每个月的销售额趋势是怎样的？',
 'output': '根据数据，2024年每月的销售额（revenue）趋势如下：\n\n| 月份 | 销售额 |\n|------|--------|\n| 2024-01 | 188,879.24 |\n| 2024-02 | 212,894.75 |\n| 2024-03 | 142,107.91 |\n| 2024-04 | 128,899.83 |\n| 2024-05 | 277,043.09 |\n| 2024-06 | 133,285.90 |\n| 2024-07 | 172,698.66 |\n| 2024-08 | 199,413.54 |\n| 2024-09 | 191,502.93 |\n| 2024-10 | 199,078.42 |\n| 2024-11 | 150,621.15 |\n| 2024-12 | 223,394.64 |\n\n**趋势分析：**\n\n1. **年初表现强劲**：1月和2月销售额较高，2月达到212,894.75。\n2. **春季下滑**：3-4月销售额明显下降，4月跌至全年最低点（128,899.83）。\n3. **5月峰值**：5月销售额大幅攀升至全年最高（277,043.09）。\n4. **年中波动**：6月再次大幅下降，随后7-10月保持在17万-20万之间的稳定水平。\n5. **年末回升**：11月有所回落，但12月大幅反弹至223,394.64，为全年第二高。\n\n整体来看，销售额呈现**波动性趋势**，没有单一的递增或递减规律。**5月**和**12月**是销售高峰期，而**3-4月**和**6月**相对较低。'}

In [25]:
# Stream the final answer token by token.
question = "每个月的销售额趋势是怎样的？"
is_answer_started = False

agent = create_pandas_dataframe_agent(llm=llm, df=df, verbose=False, allow_dangerous_code=True, agent_type="openai-tools")

async for event in agent.astream_events({"input": question}, version="v2"):
    if event["event"] == "on_tool_start":
        print(f"\n调用工具: {event['name']}")
    elif event["event"] == "on_tool_end":
        print("工具调用完成，正在生成回答...")
    elif event["event"] == "on_chat_model_stream":
        token = event["data"]["chunk"].content
        if isinstance(token, str) and token:
            if not is_answer_started:
                print("\n最终回答:")
                is_answer_started = True
            print(token, end="", flush=True)





调用工具: python_repl_ast
工具调用完成，正在生成回答...

最终回答:
根据数据，2024年各月的销售额（revenue）趋势如下：

| 月份 | 销售额 |
|------|--------|
| 2024-01 | 188,879.24 |
| 2024-02 | 212,894.75 |
| 2024-03 | 142,107.91 |
| 2024-04 | 128,899.83 |
| 2024-05 | 277,043.09 |
| 2024-06 | 133,285.90 |
| 2024-07 | 172,698.66 |
| 2024-08 | 199,413.54 |
| 2024-09 | 191,502.93 |
| 2024-10 | 199,078.42 |
| 2024-11 | 150,621.15 |
| 2024-12 | 223,394.64 |

**趋势分析：**

1. **整体波动较大**：销售额在12.9万到27.7万之间波动，没有呈现单一的上升或下降趋势。

2. **高峰期**：
   - **5月**达到全年最高峰（277,043.09）
   - **12月**次高（223,394.64）
   - **2月**也较高（212,894.75）

3. **低谷期**：
   - **4月**为全年最低（128,899.83）
   - **6月**次低（133,285.90）
   - **3月**和**11月**也相对较低

4. **季度表现**：
   - Q1（1-3月）：先升后降
   - Q2（4-6月）：波动剧烈，4月最低，5月最高
   - Q3（7-9月）：相对稳定，维持在17-20万之间
   - Q4（10-12月）：11月下滑，12月大幅反弹

总体而言，销售额呈现**周期性波动**特征，可能在某些月份有促销活动或季节性因素影响。